## Model selection — one honest screen
Every candidate run through the SAME protocol: train-only normalization stats,
the same first-80%-of-engines split (seed 42 in the models, not the split),
predict once at each TEST engine's last cycle vs NASA ground truth.
Headline metric: critical-zone [0-25] RMSE. Tiebreaker: NASA score. Also reported:
global RMSE, late %.
Flat models (mean, ridge, RF, xgboost) get the 42 engineered features; the LSTM gets
the 14 normalized sensor channels as 30-cycle sequences.
This is a FAMILY screen at fixed/untuned configs — see the asymmetry note in the
decision cell.

In [1]:
import sys
import warnings
from pathlib import Path

import pandas as pd
from IPython.display import display

sys.path.insert(0, str(Path('..').resolve() / 'src'))
from turbofan.evaluation.comparison import run_comparison, decision_summary

warnings.filterwarnings('ignore')
RAW = Path('../data/raw')
MODEL_ORDER = ['mean', 'ridge', 'rf', 'xgboost', 'lstm']

In [2]:
df = run_comparison(RAW)

FD001  mean     crit= 71.63  nasa=   30932
FD001  ridge    crit= 20.36  nasa=     843


FD001  rf       crit= 11.25  nasa=     798


FD001  xgboost  crit=  7.91  nasa=     613


FD001  lstm     crit=  3.13  nasa=     897
-- FD001 done --


FD002  mean     crit= 74.71  nasa=  153949
FD002  ridge    crit= 18.51  nasa=   18959


...

FD003  lstm     crit=  4.27  nasa=    1378
-- FD003 done --


FD004  mean     crit= 79.52  nasa=  190834
FD004  ridge    crit= 25.76  nasa=   10652


FD004  rf       crit= 11.04  nasa=    5445


FD004  xgboost  crit=  9.22  nasa=    5094


FD004  lstm     crit=  6.48  nasa=    8032
-- FD004 done --


In [3]:
df['model'] = pd.Categorical(df['model'], MODEL_ORDER, ordered=True)
df = df.sort_values(['dataset', 'model'])

table = (df.set_index(['dataset', 'model'])
           [['critical_rmse', 'nasa', 'global_rmse', 'late_pct']]
           .rename(columns={'critical_rmse': '[0-25] crit RMSE', 'nasa': 'NASA score',
                            'global_rmse': 'global RMSE', 'late_pct': 'late %'})
           .round({'[0-25] crit RMSE': 2, 'global RMSE': 2, 'late %': 1, 'NASA score': 0}))
display(table)

[0-25] crit RMSE  NASA score  global RMSE  late %
dataset model                                                     
FD001   mean                71.63     30932.0        42.85    50.0
        ridge               20.36       843.0        19.65    60.0
        rf                  11.25       798.0        18.20    64.0
        xgboost              7.91       613.0        17.47    59.0
        lstm                 3.13       897.0        17.56    62.0
FD002   mean                74.71    153949.0        54.09    55.2
        ridge               18.51     18959.0        32.05    47.9
        rf                   6.50     14215.0        28.06    48.3
        xgboost              5.64     14282.0        28.00    43.2
        lstm                 6.47      6503.0        26.16    51.0
FD003   mean                80.37     57977.0        45.07    65.0
        ridge               19.02      1564.0        21.54    61.0
        rf                   6.69      1257.0        17.25    57.0
        xgboost              5.27       758.0        16.66    49.0
        lstm                 4.27      1378.0        15.98    58.0
FD004   mean                79.52    190834.0        54.91    54.4
        ridge               25.76     10652.0        33.82    50.0
        rf                  11.04      5445.0        28.45    51.2
        xgboost              9.22      5094.0        28.11    49.2
        lstm                 6.48      8032.0        26.80    51.2

In [4]:
agg, wins = decision_summary(df)

print('Mean critical-zone RMSE / mean NASA across FD001-FD004 (sorted):')
print(agg.round(2))
print('\nDatasets won on critical-zone RMSE:')
print(wins)
print(f'\nLowest mean critical-zone RMSE: {agg.index[0]}')
print(f'Runner-up: {agg.index[1]}  '
      f'(margin {agg["mean_crit"].iloc[1] - agg["mean_crit"].iloc[0]:.2f} cycles)')

Mean critical-zone RMSE / mean NASA across FD001-FD004 (sorted):
         mean_crit  mean_nasa
model                        
lstm          5.09    4202.46
xgboost       7.01    5186.85
rf            8.87    5428.90
ridge        20.91    8004.47
mean         76.56  108423.15

Datasets won on critical-zone RMSE:
model
lstm       3
xgboost    1
mean       0
rf         0
ridge      0
Name: count, dtype: int64

Lowest mean critical-zone RMSE: lstm
Runner-up: xgboost  (margin 1.92 cycles)


In [5]:
from turbofan.evaluation.comparison import (
    run_comparison_multiseed, multiseed_summary, contender_gap,
)

# Re-run the two contenders over 5 seeds. Features + split are built once per dataset and
# reused, so only model init/training varies. (LSTM fits on FD002/FD004 dominate runtime —
# drop to seeds=(42, 7, 123) for a faster first pass.)
df_ms = run_comparison_multiseed(RAW, seeds=(42, 7, 123, 2024, 99), models=("xgboost", "lstm"))

# Banded critical-zone RMSE: mean ± std per dataset × model
display(multiseed_summary(df_ms, "critical_rmse"))

# Head-to-head: does the LSTM lead survive ±1σ on each dataset?
display(contender_gap(df_ms, a="lstm", b="xgboost"))

# Pooled across all datasets × seeds
overall = df_ms.groupby("model")["critical_rmse"].agg(["mean", "std"]).round(2)
print("Pooled critical-zone RMSE (all datasets × seeds):")
print(overall)

FD001  xgboost  seed=42    crit=  7.91
FD001  lstm     seed=42    crit=  3.13
FD001  xgboost  seed=7     crit=  8.12
FD001  lstm     seed=7     crit=  3.80
FD001  xgboost  seed=123   crit=  8.00
FD001  lstm     seed=123   crit=  3.87
FD001  xgboost  seed=2024  crit=  8.02
FD001  lstm     seed=2024  crit=  2.32
FD001  xgboost  seed=99    crit=  8.38
FD001  lstm     seed=99    crit=  2.57
-- FD001 done (5 seeds) --
FD002  xgboost  seed=42    crit=  5.64
FD002  lstm     seed=42    crit=  6.47
FD002  xgboost  seed=7     crit=  6.18
FD002  lstm     seed=7     crit=  5.01
FD002  xgboost  seed=123   crit=  5.58
FD002  lstm     seed=123   crit=  4.78
FD002  xgboost  seed=2024  crit=  5.99
FD002  lstm     seed=2024  crit=  4.35
FD002  xgboost  seed=99    crit=  5.92
FD002  lstm     seed=99    crit=  4.58
-- FD002 done (5 seeds) --
FD003  xgboost  seed=42    crit=  5.27
FD003  lstm     seed=42    crit=  4.27
FD003  xgboost  seed=7     crit=  5.59
FD003  lstm     seed=7     crit=  2.81
FD003  xgb

mean   std   min   max
dataset model                          
FD001   lstm     3.14  0.70  2.32  3.87
        xgboost  8.09  0.18  7.91  8.38
FD002   lstm     5.04  0.84  4.35  6.47
        xgboost  5.86  0.25  5.58  6.18
FD003   lstm     3.77  0.68  2.81  4.44
        xgboost  5.10  0.36  4.62  5.59
FD004   lstm     6.64  0.67  5.93  7.74
        xgboost  8.61  0.42  8.14  9.22

,lstm_mean,lstm_std,xgboost_mean,xgboost_std,gap_b_minus_a,separated_1sigma
dataset,,,,,,
FD001,3.14,0.70,8.09,0.18,4.95,True
FD002,5.04,0.84,5.86,0.25,0.83,False
FD003,3.77,0.68,5.10,0.36,1.33,True
FD004,6.64,0.67,8.61,0.42,1.98,True


Pooled critical-zone RMSE (all datasets × seeds):
         mean   std
model              
lstm     4.65  1.53
xgboost  6.92  1.54


## Decision

**Ship the LSTM** (14 normalized sensor channels, 30-cycle sequences). Honest claim:
**best among these candidates under this protocol** — last-cycle prediction vs NASA
ground truth, headline = critical-zone [0-25] RMSE, NASA score as tiebreaker.

**Evidence (5 seeds, model init/training varied; split held fixed):**
- Pooled critical-zone RMSE: LSTM 4.65 vs XGBoost 6.92.
- LSTM separated beyond ±1σ on FD001, FD003, FD004 — on these its worst seed beats
  XGBoost's best seed (non-overlapping ranges).
- FD002 is within noise (LSTM 5.04±0.84 vs XGBoost 5.86±0.25); the single-seed XGBoost
  "win" there was a seed-42 artifact and reverses to a slight LSTM edge over five seeds.
- XGBoost is meaningfully better on no dataset.

**Why this beats the obvious objection:** the LSTM won *untuned*. XGBoost's 20-trial
per-dataset Optuna search (nb02) moved mean critical RMSE by ~0.04 cycles — it targeted
global validation RMSE, not the critical zone — so its tuning edge on the headline metric
is empirically ~zero. Tuning the LSTM would likely widen, not close, the gap.

**What was not tested / known characteristics:** train/val split composition variance
(only model seed was varied); per-model HP search beyond XGBoost's (mis-targeted) one;
one architecture per family; one RUL cap (125) and window (30). The LSTM is noisier
run-to-run than XGBoost (std up to 0.84 vs 0.42), and on FD001 carries a worse NASA score
than XGBoost despite winning critical RMSE (fatter tails outside the critical zone).